We start by loading the data into a structured form containing the symptoms, the disease, and the alternatives (the true symptom if one if the alternatives).

In [30]:
import random
import csv



data = []
with open("symptom_dataset.csv", newline="") as file:
    rows = list(csv.DictReader(file))

diseases = list({row["Disease"].strip() for row in rows})
data = []
for row in rows:
    answer = row["Disease"].strip()
    symptoms = [symptom.strip() for column, symptom in row.items() if column.startswith("Symptom_") and symptom.strip()]
    alternatives = random.sample([disease for disease in diseases if disease != answer], 4)
    alternatives.append(answer)
    data.append({
        "symptoms": symptoms,
        "alternatives": alternatives,
        "answer": answer,
    })


print(diseases)
print(data)
for item in data:
    print(item["answer"])
    print(item["alternatives"])

['Fungal infection', 'GERD', 'Heart attack', 'Bronchial Asthma', 'Arthritis', 'Malaria', 'Hypothyroidism', 'Paralysis (brain hemorrhage)', 'Chicken pox', 'Alcoholic hepatitis', 'Hepatitis D', 'Hepatitis B', 'Drug Reaction', 'Allergy', 'Hepatitis E', 'hepatitis A', 'Acne', 'Psoriasis', 'Osteoarthristis', 'Chronic cholestasis', 'Urinary tract infection', 'Hypertension', 'Hyperthyroidism', 'Peptic ulcer diseae', 'Jaundice', '(vertigo) Paroymsal  Positional Vertigo', 'Pneumonia', 'Diabetes', 'Dengue', 'Gastroenteritis', 'Varicose veins', 'Dimorphic hemmorhoids(piles)', 'Tuberculosis', 'Impetigo', 'Common Cold', 'Migraine', 'Hypoglycemia', 'AIDS', 'Typhoid', 'Cervical spondylosis', 'Hepatitis C']
[{'symptoms': ['itching', 'skin_rash', 'nodal_skin_eruptions', 'dischromic _patches'], 'alternatives': ['Varicose veins', 'Cervical spondylosis', 'Hepatitis E', 'Heart attack', 'Fungal infection'], 'answer': 'Fungal infection'}, {'symptoms': ['skin_rash', 'nodal_skin_eruptions', 'dischromic _patche

Now we generate, using an LLM, the universal argument tree that defines the lingua france.
These arguments and relations are the only argument and relations that the agents will be able to use.
The agent will still be responsible for assigning a strength to the arguments and deciding which arguments to disclose in the debate.

In [42]:
import random

from pydantic import BaseModel, ConfigDict
from autom8 import Agent
from mtax import Argument, BipolarMultitree, Relation


NUM_ARGUMENTS = 5
DEPTH = 3
EXPANSION_PROBABILITY = 0.5
SEED = 0


class GeneratedArguments(BaseModel):
    model_config = ConfigDict(extra="forbid")
    support: list[str]
    attack: list[str]



def argument_response_format(support_count, attack_count):
    schema = GeneratedArguments.model_json_schema()
    for name, count in (("support", support_count), ("attack", attack_count)):
        schema["properties"][name]["minItems"] = count
        schema["properties"][name]["maxItems"] = count
    return {
        "type": "json_schema",
        "json_schema": {
            "name": "generated_arguments",
            "schema": schema,
            "strict": True,
        },
    }


agent = Agent(
    model="openai/gpt-4o-mini",
    tool_registry={},
    max_completion_tokens=10000,
    system_prompt=(
        "You construct concise, evidence-grounded arguments for a diagnostic debate. "
        "Use only the supplied symptoms and candidate diagnoses. "
        "Do not claim that an unobserved symptom is absent."
    ),
)


def generate(prompt, support_count, attack_count):
    response = agent.invoke(prompt, response_format=argument_response_format(support_count, attack_count))
    agent.reset(0)
    return GeneratedArguments.model_validate_json(response)


def add_argument(tree, arguments, text, parent, kind):
    label = f"argument_{len(arguments)}"
    arguments[label] = Argument(label=label, text=text)
    tree.add_relation(label, parent, kind)
    return label


def build_argument_tree(item, seed=SEED):
    random_generator = random.Random(seed)
    topics = item["alternatives"]
    symptoms = ", ".join(item["symptoms"])
    candidates = ", ".join(topics)

    tree = BipolarMultitree(topics=set(topics))
    arguments = {topic: Argument(label=topic, text=topic) for topic in topics}
    leaves = []

    for topic in topics:
        generated = generate(f"""
                                Symptoms: {symptoms}
                                Candidate diagnoses: {candidates}
                                Target diagnosis: {topic}

                                Generate exactly {NUM_ARGUMENTS} arguments supporting the target diagnosis
                                and exactly {NUM_ARGUMENTS} arguments attacking it.
                                """, NUM_ARGUMENTS, NUM_ARGUMENTS)

        if len(generated.support) != NUM_ARGUMENTS or len(generated.attack) != NUM_ARGUMENTS:
            raise ValueError("LLM returned the wrong number of direct arguments")

        for text in generated.support:
            leaves.append(add_argument(tree, arguments, text, topic, "support"))
        for text in generated.attack:
            leaves.append(add_argument(tree, arguments, text, topic, "attack"))

    for _ in range(DEPTH - 1):
        planned = {"support": [], "attack": []}
        next_leaves = []
        for parent in leaves:
            if random_generator.random() >= EXPANSION_PROBABILITY:
                next_leaves.append(parent)
                continue
            planned[random_generator.choice(("support", "attack"))].append(parent)
        for kind, parents in planned.items():
            if not parents:
                continue
            parent_arguments = "\n".join(f"{index + 1}. {arguments[parent].text}"
                                         for index, parent in enumerate(parents))
            generated = generate(f"""
                                    Symptoms: {symptoms}
                                    Candidate diagnoses: {candidates}

                                    Generate one argument for each parent argument below.
                                    Each new argument must {kind} its parent argument.
                                    Return the new texts in the {kind} list in the same order as the parents.
                                    Return an empty list for the other relation type.

                                    Parent arguments:
                                    {parent_arguments}
                                    """,
                                 len(parents) if kind == "support" else 0,
                                 len(parents) if kind == "attack" else 0)

            child_arguments = generated.support if kind == "support" else generated.attack
            if len(child_arguments) != len(parents):
                raise ValueError("LLM returned the wrong number of child arguments")
            for parent, text in zip(parents, child_arguments):
                next_leaves.append(add_argument(tree, arguments, text, parent, kind))
        leaves = next_leaves
    return tree, arguments


def private_subtree(tree, arguments, size, seed):
    if size < len(tree.topics):
        raise ValueError("size must include every topic")

    random_generator = random.Random(seed)
    selected = set(tree.topics)
    selected_relations = []
    while len(selected) < size:
        frontier = [relation for relation in tree.relations if relation[0] not in selected and relation[1] in selected]
        if not frontier:
            break
        source, target, kind = random_generator.choice(sorted(frontier))
        selected.add(source)
        selected_relations.append(Relation(source=source, target=target, kind=kind))
    for source, target, kind in tree.relations:
        if source in selected and target in selected:
            relation = Relation(source=source, target=target, kind=kind)
            if relation not in selected_relations:
                selected_relations.append(relation)
    private_arguments = {
        label: arguments[label]
        for label in selected - tree.topics
    }
    return private_arguments, selected_relations


universal_trees = []
for i in range(20):
    universal_trees.append(build_argument_tree(data[i]))

universal_tree, universal_arguments = universal_trees[0]
agent_1_arguments, agent_1_relations = private_subtree(universal_tree, universal_arguments, size=20, seed=1)
agent_2_arguments, agent_2_relations = private_subtree(universal_tree, universal_arguments, size=20, seed=2)
print(f"Universal arguments: {len(universal_arguments)}")
print(f"Agent 1 private arguments: {len(agent_1_arguments)}")
print(f"Agent 2 private arguments: {len(agent_2_arguments)}")

ValidationError: 1 validation error for GeneratedArguments
  Invalid JSON: EOF while parsing a list at line 9636 column 0 [type=json_invalid, input_value='{"support":[],"attack":[...     \n\n          \n\n', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/json_invalid

Below we print some basic examples from the generated trees.

In [32]:

universal_tree, universal_arguments = universal_trees[1]

for argument in universal_arguments.values():
    print(argument.label, ":", argument.text)

for source, target, kind in sorted(universal_tree.relations):
    print(f"{universal_arguments[source].label} --{kind}--> {universal_arguments[target].label}")
    # print(f"{universal_arguments[source].text} --{kind}--> {universal_arguments[target].text}")

Hepatitis D : Hepatitis D
Gastroenteritis : Gastroenteritis
Diabetes : Diabetes
Dimorphic hemmorhoids(piles) : Dimorphic hemmorhoids(piles)
Fungal infection : Fungal infection
argument_5 : Skin rashes are commonly observed in patients with Hepatitis D, particularly in the context of chronic liver disease.
argument_6 : Nodal skin eruptions can occur due to immune-mediated responses in Hepatitis D, linking the skin symptoms to liver pathology.
argument_7 : Dyschromic patches can be a result of liver dysfunction, which is consistent with Hepatitis D, where liver damage disrupts normal skin pigmentation.
argument_8 : Hepatitis D infection often coincides with other hepatitis infections, and skin manifestations are common in viral hepatitis cases, indicating a systemic effect on the body.
argument_9 : The presence of skin symptoms alongside liver enzyme abnormalities suggests a connection to Hepatitis D as a possible primary cause.
argument_10 : Gastroenteritis can also present with skin ra

At this point we have parsed the data and created a set of universal trees (defining the debate lingua franca).
We now instantiate 2 agents and let them debate the ranking of the alternatives.
If the debate is resolved, we look at the alternative ranked the highest, and consider that the answer/recommendation.
We then validate this against the true answer.
If the debate is unresolved after max_rounds, we consider the recommendation to be wrong.

In [41]:
import json
from pydantic import BaseModel, ConfigDict, Field

from autom8 import Agent
from mtax import Argument, MTAXAgent, ExchangeConfig, MTAX
from mtax.disclosure_measure import ranking_disclosure_effects
from mtax.schema import Disclosure, Pass


class Strength(BaseModel):
    model_config = ConfigDict(extra="forbid")
    score: float = Field(ge=0, le=1)

strength_response_format = {
    "type": "json_schema",
    "json_schema": {
        "name": "strength",
        "schema": Strength.model_json_schema(),
        "strict": True,
    },
}


class LLMAgent(MTAXAgent):
    def __init__(self, name, symptoms, topics, private_arguments, private_relations):
        self.llm = Agent(model="openai/gpt-4o-mini", tool_registry={}, max_completion_tokens=10000)
        self.symptoms = ", ".join(symptoms)
        self.candidate_topics = ", ".join(topics)
        private_strengths = {topic: self.rate(Argument(label=topic, text=topic)) for topic in topics}
        private_strengths.update({label: self.rate(argument) for label, argument in private_arguments.items()})
        super().__init__(name=name, private_arguments=private_arguments,
                         private_relations=private_relations, private_strengths=private_strengths)


    def contribute(self, public_bm, violation_feedback=None):
        """Logic for disclosing relations/arguments."""
        available = [relation for relation, _ in self.available_relations(public_bm)]
        if not available:
            return Pass(action="pass")
        effects = ranking_disclosure_effects(self.build_qbaf_from_bm(public_bm), self.private_qbaf, self.topics, relations=available)
        if not effects or effects[0][1] < 0:
            return Pass(action="pass")
        relation = effects[0][0]
        arguments = ()
        if relation.source not in public_bm.arguments:
            arguments = (self.private_arguments.get(relation.source, Argument(label=relation.source, text=relation.source)),)
        return Disclosure(arguments=arguments, relations=(relation,))


    def rate(self, argument):
        response = self.llm.invoke(
            f"""
                Symptoms: {self.symptoms}
                Candidate diagnoses: {self.candidate_topics}
                Argument: {argument.text}

                How credible is this argument for this diagnostic case?
                Return 0 for unsupported and 1 for strongly supported.
                """,
                response_format=strength_response_format)
        self.llm.reset(0)
        score = float(json.loads(response)["score"])
        if not 0 <= score <= 1:
            raise ValueError("LLM returned a strength outside [0, 1]")
        return score


debate_results = []
for i in range(20):
    item = data[i]
    universal_tree, universal_arguments = universal_trees[i]
    
    agent_1_arguments, agent_1_relations = private_subtree(universal_tree, universal_arguments, size=20, seed=1)
    agent_1 = LLMAgent("agent_1", item["symptoms"], item["alternatives"], agent_1_arguments, agent_1_relations)
    
    agent_2_arguments, agent_2_relations = private_subtree(universal_tree, universal_arguments, size=20, seed=2)
    agent_2 = LLMAgent("agent_2", item["symptoms"], item["alternatives"], agent_2_arguments, agent_2_relations)
    
    exchange = MTAX(
        agents=[agent_1, agent_2],
        topics=item["alternatives"],
        config=ExchangeConfig(max_rounds=50),
    )


    for state in exchange:
        # print(state.round_index)
        # print(state.agent_statuses)
        continue
    result = exchange.result()
    prediction = None

    if result.resolved:
        prediction = max(item["alternatives"], key=agent_1.private_qbaf.final_strength)
    debate_results.append({
        "answer": item["answer"],
        "prediction": prediction,
        "resolved": result.resolved,
        "correct": result.resolved and prediction == item["answer"],
    })
    print(result.resolved)
    print(result.termination_reason)

accuracy = sum(result["correct"] for result in debate_results) / len(debate_results)
resolution_rate = sum(result["resolved"] for result in debate_results) / len(debate_results)
print(f"Accuracy: {accuracy:.1%}")
print(f"Resolution rate: {resolution_rate:.1%}")

True
resolved
True
resolved
True
resolved
True
resolved
True
resolved


IndexError: list index out of range

We now compare the debate accuracy to the accuracy achieved by a single agent in a non-debate environment.

In [40]:
from pydantic import BaseModel, ConfigDict
from autom8 import Agent


class Diagnosis(BaseModel):
    model_config = ConfigDict(extra="forbid")
    disease: str

def diagnosis_response_format(alternatives):
    schema = Diagnosis.model_json_schema()
    schema["properties"]["disease"]["enum"] = alternatives
    return {
        "type": "json_schema",
        "json_schema": {
            "name": "diagnosis",
            "schema": schema,
            "strict": True,
        },
    }



classifier = Agent(
    model="openai/gpt-4o-mini",
    tool_registry={},
    max_completion_tokens=1000,
)

results = []
for i in range(20):
    item = data[i]
    response = classifier.invoke(
        f"""
            Symptoms: {", ".join(item["symptoms"])}

            Possible diagnoses: {", ".join(item["alternatives"])}

            Which diagnosis best explains the symptoms?
            """,
            response_format=diagnosis_response_format(item["alternatives"]))
    classifier.reset(0)
    prediction = Diagnosis.model_validate_json(response).disease
    if prediction not in item["alternatives"]:
        raise ValueError(f"Unknown diagnosis: {prediction}")

    results.append({
        "answer": item["answer"],
        "prediction": prediction,
        "correct": prediction == item["answer"],
    })
accuracy = sum(result["correct"] for result in results) / len(results)
print(f"Accuracy: {accuracy:.1%}")

Accuracy: 80.0%
